In [1]:
# Choose a theme to your convenience
PLOTLY_THEME = "plotly_dark"
#PLOTLY_THEME = "plotly_white"


In [2]:
from dataclasses import dataclass, asdict

@dataclass
class SimulationParameters:
    re: float
    N: int
    steps: int
    sampling: int
    K: float
    dt: float
    pressure_iters: int
    precond_steps: int
    precond_dt: float
    full_solve_iters: int
    scheme: str
    seed: int

In [3]:
import configparser
def write_config(params):
    config = configparser.ConfigParser()
    config["Simulation"] = asdict(params)
    with open('simulation.ini', 'w') as configfile:
      config.write(configfile)

In [4]:
import subprocess,  os

def launch_c():
    subprocess.run(["cmake", "--build", "build"])
    
    with open(os.devnull, "w") as devnull:
        proc = subprocess.Popen(
            ["./build/cldc"],
            stdout=devnull,
            stderr=devnull
        )

In [5]:
import mmap
import struct
import time

from tqdm.notebook import tqdm

import numpy as np
import plot_run
import posix_ipc

def launch_run(params, graph_out=True):
    FLAT_ARRAY_SIZE = params.N * params.N * 3
    record_size = params.steps // params.sampling
    output_array = np.zeros((record_size, params.N, params.N, 3))
    
    shm = posix_ipc.SharedMemory("/sim_shm", posix_ipc.O_CREAT, size=8 + 8 * FLAT_ARRAY_SIZE)
    mm = mmap.mmap(shm.fd, shm.size)
    
    shm.close_fd()
    
    mm.write(struct.pack("Q", 0))
    
    write_idx = np.frombuffer(mm, dtype=np.uint64, count=1, offset=0)
    data = np.frombuffer(mm, dtype=np.float64, count=FLAT_ARRAY_SIZE, offset=8)
    
    plotter = plot_run.RunPlot(params.re, params.K, params.N, PLOTLY_THEME)
    plotter.prepare()
    last = 0
    try:
        pbar = tqdm(total=params.steps)
        pbar.set_description("Waiting for C simulation")
        launch_c()
        while True:
            w = int(write_idx[0])
            index = w // params.sampling
            if w > last:
                pbar.set_description("")
                pbar.update(params.sampling)
                last = w
                velocity = data[:params.N * params.N * 2].copy().reshape(params.N, params.N, 2)
                pressure = data[params.N * params.N * 2:params.N * params.N * 3].copy().reshape(params.N, params.N, 1)
                frame = np.concatenate([velocity, pressure], axis=-1)
                output_array[index] = frame
                plotter.update(velocity, pressure, index)
            if index >= record_size - 1:
                break
            time.sleep(0.00001)
    except KeyboardInterrupt:
        print("User stopped run")
    print("Run done")
    # np.save("runs/out", output_array)
    return output_array


## Init storage

In [81]:
import os
with open(os.path.expanduser("~") + "/.keys") as f:
    for line in f:
        if line.startswith("export "):
            key, val = line.strip().split("=", 1)
            os.environ[key.replace("export ","")] = val


import boto3
import zarr
import s3fs

X = 128
T = 50
C_in = 4
C_out = 3

fs = s3fs.S3FileSystem(
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
    client_kwargs={"endpoint_url": "http://localhost:8333"}
)

store_X = s3fs.S3Map(root="ldcdataset/X", s3=fs, check=False)
store_Y = s3fs.S3Map(root="ldcdataset/Y", s3=fs, check=False)

def init_bucket():
    s3 = boto3.client(
        "s3",
        endpoint_url="http://localhost:8333",
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    )
    s3.create_bucket(Bucket="ldcdataset")


def init_dataset(): 
    zarr.open(store_X, mode="w", shape=(0, X, X, C_in), chunks=(1, X, X, C_in), dtype='f8')
    zarr.open(store_Y, mode="w", shape=(0, T, X, X, C_out), chunks=(1, X, X, T, C_out), dtype='f8')

# init_dataset()

dts_X = zarr.open(store_X, mode="a")
dts_Y = zarr.open(store_Y, mode="a")

In [82]:
dts_X.shape

(19, 128, 128, 4)

In [52]:

dt = 1e-3
sr = 50
prec = 150
steps = 10150
t_start = 5
t_end = 10
t_inter = int((t_end - t_start) / T / dt / sr)
tmid_idx = int(t_start / t_end * steps / sr)
tlast_idx = tmid_idx + T * t_inter

In [77]:
res = [i for i in range(100, 1001, 50)]
param_set = [SimulationParameters(re, 128, steps, sr, 1.0, dt, 5, prec, 1e-4, 0, "RK4", re) for re in res]

In [78]:


def param_to_entry(param):
    write_config(param)
    nu = 1.0 / param.re
    out = launch_run(param)
    # out of shape iter / sr, X, X, 3 (u v p)
    
    # data x : X, X, C with C: icu0, icu1, icp, nu
    
    data_x = np.concatenate([out[tmid_idx], (np.ones((X, X)) * nu)[..., None]], axis=-1)
    # data y : T, X, X, C with C: u0, u1, p with t=0 t0idx
    data_y = out[tmid_idx:tlast_idx:t_inter]
    dts_X.append(data_x[None, ...])
    dts_Y.append(data_y[None, ...])

In [80]:
for param in param_set[1:]:
    param_to_entry(param)


FigureWidget({
    'data': [{'line': {'width': 2},
              'mode': 'lines',
              'name': 'min',
              'type': 'scatter',
              'uid': 'b2333353-123b-4f49-9fcd-0bc63c2e7cc8',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'mean',
              'type': 'scatter',
              'uid': 'd4688da6-522a-4888-b908-b7b599137ddf',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'max',
              'type': 'scatter',
              'uid': '2ea10008-3db2-4e5e-bbd4-6cde7dc5c55c',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'profile',
              'type': 'scatter',
              'uid': 'e62ea6dd-0bbf-4537-

FigureWidget({
    'data': [{'colorbar': {'title': {'text': 'Velocity norm'}, 'x': 0.45},
              'colorscale': [[0.0, '#440154'], [0.1111111111111111, '#482878'],
                             [0.2222222222222222, '#3e4989'], [0.3333333333333333,
                             '#31688e'], [0.4444444444444444, '#26828e'],
                             [0.5555555555555556, '#1f9e89'], [0.6666666666666666,
                             '#35b779'], [0.7777777777777778, '#6ece58'],
                             [0.8888888888888888, '#b5de2b'], [1.0, '#fde725']],
              'type': 'heatmap',
              'uid': 'd307347f-22e4-4372-928a-6fa8d07f65b7',
              'xaxis': 'x',
              'yaxis': 'y',
              'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAA='),
                    'dtype': 'f8',
                    'shape': '128, 128'},
              'zsmooth': False},
             {'colorbar': {'title': {'text': 'Pressure'}, 'x': 1.0},
    

  0%|          | 0/10150 [00:00<?, ?it/s]

[100%] Built target cldc
Run done


FigureWidget({
    'data': [{'line': {'width': 2},
              'mode': 'lines',
              'name': 'min',
              'type': 'scatter',
              'uid': '45473183-afb4-4b00-8ebe-bae4ed34307e',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'mean',
              'type': 'scatter',
              'uid': 'a69b7a0f-ffcc-4b50-bdc8-2ad5a1fbcad5',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'max',
              'type': 'scatter',
              'uid': 'a9f53e57-64cf-497e-b651-4940cb1c9a34',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'profile',
              'type': 'scatter',
              'uid': 'd19f76e7-15f3-4f75-

...

[100%] Built target cldc
Run done


FigureWidget({
    'data': [{'line': {'width': 2},
              'mode': 'lines',
              'name': 'min',
              'type': 'scatter',
              'uid': '4722ff8b-39ba-49c1-bcb6-1f8a75992820',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'mean',
              'type': 'scatter',
              'uid': '78eea263-acd6-404f-9826-aae4beef778d',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'max',
              'type': 'scatter',
              'uid': '01fa8593-a072-4113-888f-1aa1dd6b255d',
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'width': 2},
              'mode': 'lines',
              'name': 'profile',
              'type': 'scatter',
              'uid': '33b49b5e-ffd8-4357-

FigureWidget({
    'data': [{'colorbar': {'title': {'text': 'Velocity norm'}, 'x': 0.45},
              'colorscale': [[0.0, '#440154'], [0.1111111111111111, '#482878'],
                             [0.2222222222222222, '#3e4989'], [0.3333333333333333,
                             '#31688e'], [0.4444444444444444, '#26828e'],
                             [0.5555555555555556, '#1f9e89'], [0.6666666666666666,
                             '#35b779'], [0.7777777777777778, '#6ece58'],
                             [0.8888888888888888, '#b5de2b'], [1.0, '#fde725']],
              'type': 'heatmap',
              'uid': 'a6312a42-5cd7-4d70-812b-fd3ec49a5ce1',
              'xaxis': 'x',
              'yaxis': 'y',
              'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAA='),
                    'dtype': 'f8',
                    'shape': '128, 128'},
              'zsmooth': False},
             {'colorbar': {'title': {'text': 'Pressure'}, 'x': 1.0},
    

  0%|          | 0/10150 [00:00<?, ?it/s]

[100%] Built target cldc
Run done
